# PortfolioRiskDiagnosis Review

This notebook is a review surface for the current diagnosis engine.

It is organized around five user-facing questions:

1. Overall Diagnosis Summary
2. Top Risk Categories (Ranked)
3. Top Risk Drivers
4. Supporting Evidence
5. Confidence and Completeness Flags

The goal is to make the diagnosis understandable enough for product review, UI brainstorming, and trust checks.


## Philosophy

This notebook is intentionally object-first.

We still use dataframes for display, but the logic flows through the typed `PortfolioRiskDiagnosis` object from [diagnosis.py](/Users/sagnikrana/Documents/GitHub/portfolio-analyzer/portfolio_analyzer/diagnosis.py).

That keeps the review aligned with the architecture we actually want to build forward from.


In [ ]:
from pathlib import Path
import importlib
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DIAGNOSIS_DIR = ROOT / 'data' / 'processed' / 'diagnosis'
OUTPUT_PATH = DIAGNOSIS_DIR / 'portfolio_risk_diagnosis_enriched.json'

import portfolio_analyzer.diagnosis as diagnosis_module
importlib.reload(diagnosis_module)

portfolio_risk_diagnosis_from_saved_artifacts = diagnosis_module.portfolio_risk_diagnosis_from_saved_artifacts
DiagnosisConcern = diagnosis_module.DiagnosisConcern

diagnosis = portfolio_risk_diagnosis_from_saved_artifacts(DIAGNOSIS_DIR)

print('Diagnosis directory:', DIAGNOSIS_DIR)
print('Run ID:', diagnosis.run_id)
print('Observed risk:', diagnosis.observed_risk_score, diagnosis.observed_risk_band)
print('Stated risk:', diagnosis.stated_risk_score, diagnosis.stated_risk_band)
print('Confidence:', diagnosis.confidence_band)


In [ ]:
def confidence_interpretation(diagnosis):
    coverage = diagnosis.data_coverage.model_dump()
    available_count = sum(1 for value in coverage.values() if value)
    if diagnosis.confidence_band == 'High' and available_count >= 7:
        return 'High confidence: multiple independent signals agree and coverage is broad.'
    if diagnosis.confidence_band in {'High', 'Medium'}:
        return 'Medium confidence: several signals agree, but some areas still need deeper modeling.'
    return 'Low confidence: data is limited or the diagnosis remains preliminary.'


def build_overall_summary_df(diagnosis):
    return pd.DataFrame(
        [
            {
                'observed_risk_score': diagnosis.observed_risk_score,
                'observed_risk_band': diagnosis.observed_risk_band,
                'stated_risk_score': diagnosis.stated_risk_score,
                'stated_risk_band': diagnosis.stated_risk_band,
                'alignment_status': diagnosis.alignment,
                'overall_confidence': diagnosis.confidence_band,
                'confidence_interpretation': confidence_interpretation(diagnosis),
                'analysis_start': diagnosis.analysis_start,
                'analysis_end': diagnosis.analysis_end,
                'benchmark_symbol': diagnosis.benchmark_symbol,
            }
        ]
    )


def derive_sector_crowding_row(diagnosis):
    if not diagnosis.top_sector_drivers:
        return {
            'category': 'Sector crowding',
            'category_score': 0.0,
            'score_origin': 'derived',
            'why_it_matters': 'No dominant sector crowding signal was available.',
        }
    top_sector = diagnosis.top_sector_drivers[0]
    weight_pct = top_sector.weight_pct or 0.0
    category_score = min(100.0, round(weight_pct * 100 + (10 if 'large sector concentration' in top_sector.driver_reasons else 0), 1))
    return {
        'category': 'Sector crowding',
        'category_score': category_score,
        'score_origin': 'derived',
        'why_it_matters': f"{top_sector.sector} carries about {weight_pct:.1%} of current capital, which makes one sector dominate portfolio behavior.",
    }


def derive_company_specific_risk_row(diagnosis):
    risky_fundamental_count = sum(1 for item in diagnosis.holding_fundamentals if any('negative' in signal or 'large share of assets' in signal for signal in item.signals))
    narrative_count = len(diagnosis.narrative_evidence)
    high_beta_count = sum(1 for item in diagnosis.holding_fundamentals if (item.beta or 0.0) > 1.2)
    category_score = min(100.0, round(risky_fundamental_count * 18 + narrative_count * 4 + high_beta_count * 6, 1))
    return {
        'category': 'Company-specific risk',
        'category_score': category_score,
        'score_origin': 'derived',
        'why_it_matters': f"Narrative evidence exists for {narrative_count} key driver documents and {high_beta_count} main drivers still show above-market beta or other fragility signals.",
    }


def derive_macro_sensitivity_row(diagnosis):
    if diagnosis.macro_context is None:
        return {
            'category': 'Macro sensitivity',
            'category_score': 0.0,
            'score_origin': 'derived',
            'why_it_matters': 'Macro regime data was not available.',
        }
    market_concern = next((item for item in diagnosis.top_concerns if item.concern_key == 'market'), None)
    base = (market_concern.severity_score if market_concern else 0.0) * 0.45
    macro_score = min(100.0, round(base + len(diagnosis.macro_context.regime_flags) * 8, 1))
    return {
        'category': 'Macro sensitivity',
        'category_score': macro_score,
        'score_origin': 'derived',
        'why_it_matters': diagnosis.macro_context.summary,
    }


def build_top_risk_categories_df(diagnosis):
    native_rows = [
        {
            'category': concern.label,
            'category_score': concern.severity_score,
            'score_origin': f"base {concern.base_severity_score:.1f} + external {concern.external_adjustment_score:.1f}",
            'why_it_matters': concern.summary,
        }
        for concern in diagnosis.top_concerns
    ]
    derived_rows = [
        derive_sector_crowding_row(diagnosis),
        derive_company_specific_risk_row(diagnosis),
        derive_macro_sensitivity_row(diagnosis),
    ]
    categories_df = pd.DataFrame(native_rows + derived_rows)
    return categories_df.sort_values('category_score', ascending=False).reset_index(drop=True)


def build_top_risk_drivers_df(diagnosis):
    rows = []
    for concern in diagnosis.top_concerns:
        for ticker in concern.related_tickers:
            rows.append(
                {
                    'driver_type': 'holding',
                    'driver_name': ticker,
                    'linked_concern': concern.label,
                    'evidence': '; '.join(concern.adjustment_reasons[:2]) or concern.summary,
                }
            )
        for sector in concern.related_sectors:
            rows.append(
                {
                    'driver_type': 'sector',
                    'driver_name': sector,
                    'linked_concern': concern.label,
                    'evidence': concern.summary,
                }
            )

    if not rows:
        for driver in diagnosis.top_holding_drivers:
            rows.append(
                {
                    'driver_type': 'holding',
                    'driver_name': driver.ticker,
                    'linked_concern': 'Fallback driver',
                    'evidence': '; '.join(driver.driver_reasons),
                }
            )
    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


def build_supporting_evidence_frames(diagnosis):
    metrics_df = pd.DataFrame([item.model_dump() for item in diagnosis.supporting_metrics])
    metrics_df = metrics_df.sort_values('score', ascending=False).reset_index(drop=True)

    sector_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_sector_drivers])
    narrative_df = pd.DataFrame([item.model_dump() for item in diagnosis.narrative_evidence])
    macro_df = pd.DataFrame([diagnosis.macro_context.model_dump()]) if diagnosis.macro_context else pd.DataFrame()

    return metrics_df, sector_df, narrative_df, macro_df


def build_confidence_frames(diagnosis):
    confidence_df = pd.DataFrame(
        [
            {
                'overall_confidence': diagnosis.confidence_band,
                'confidence_interpretation': confidence_interpretation(diagnosis),
                'coverage_available_sources': sum(1 for value in diagnosis.data_coverage.model_dump().values() if value),
            }
        ]
    )
    coverage_df = pd.DataFrame(
        [
            {'source_flag': key, 'available': value}
            for key, value in diagnosis.data_coverage.model_dump().items()
        ]
    )
    gaps_df = pd.DataFrame({'evidence_gap': diagnosis.evidence_gaps})
    return confidence_df, coverage_df, gaps_df


## 1. Overall Diagnosis Summary

This section should answer the user's first trust question:

- What risk did I say I was comfortable with?
- What risk does my portfolio actually look like?
- Are those aligned?
- How confident is the system in that conclusion?


In [ ]:
overall_summary_df = build_overall_summary_df(diagnosis)
display(overall_summary_df)
display(Markdown(f"**Diagnostic summary:** {diagnosis.diagnostic_summary}"))


## 2. Top Risk Categories (Ranked)

The diagnosis should rank what kind of risk is dominant, not just dump scores.

This view combines:
- native diagnosis concerns from the object
- derived sector crowding view
- derived company-specific risk view
- derived macro sensitivity view


In [ ]:
top_risk_categories_df = build_top_risk_categories_df(diagnosis)
display(top_risk_categories_df)


## 3. Top Risk Drivers

This section names what is actually causing the risk:

- specific holdings
- dominant sectors
- the reasoning attached to those drivers


In [ ]:
top_risk_drivers_df = build_top_risk_drivers_df(diagnosis)
holding_drivers_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_holding_drivers])
sector_drivers_df = pd.DataFrame([item.model_dump() for item in diagnosis.top_sector_drivers])

display(top_risk_drivers_df)
display(holding_drivers_df)
display(sector_drivers_df)


## 4. Supporting Evidence

This is not raw-data overload. It is the minimum evidence necessary to support the diagnosis:

- benchmark-relative risk metrics
- sector exposure evidence
- filing/news evidence summaries
- macro regime context


In [ ]:
supporting_metrics_df, sector_evidence_df, narrative_evidence_df, macro_context_df = build_supporting_evidence_frames(diagnosis)
holding_fundamentals_df = pd.DataFrame([item.model_dump() for item in diagnosis.holding_fundamentals])

display(supporting_metrics_df[['metric_key', 'group', 'label', 'raw_value', 'score', 'score_readout']])
display(sector_evidence_df)
display(holding_fundamentals_df)
display(narrative_evidence_df)
display(macro_context_df)


## 5. Confidence and Completeness Flags

A financial diagnosis should never overstate certainty.

This section makes explicit:
- the overall confidence level
- how broad the source coverage is
- where the current diagnosis still has blind spots


In [ ]:
confidence_df, coverage_df, gaps_df = build_confidence_frames(diagnosis)
display(confidence_df)
display(coverage_df)
display(gaps_df)


## Save Current Enriched Diagnosis

This writes the current object view to disk so we can compare revisions over time.


In [ ]:
OUTPUT_PATH.write_text(json.dumps(diagnosis.model_dump(), indent=2) + '\n', encoding='utf-8')
print('Saved enriched diagnosis object to:')
print(OUTPUT_PATH)


## Review Prompts

When reviewing this notebook, useful questions are:

- Do the ranked categories feel believable?
- Are the named risk drivers actually the right ones?
- Is the supporting evidence too thin, too noisy, or just right?
- Does the confidence section feel appropriately humble?
- What should be surfaced directly in the UI, and what should remain secondary?
